In [33]:
import pandas as pd

# CSV faylni yuklash
data = pd.read_csv("merged.csv")

# Birinchi 5 qatorni tekshirish
print(data.head())



   credit_score  oldest_credit_line_age  oldest_account_age_months  \
0           696                    22.0                      264.0   
1           659                     3.5                       42.0   
2           662                     0.0                        0.0   
3           676                     9.0                      108.0   
4           678                     8.0                       96.0   

   num_credit_accounts  num_inquiries_6mo  account_diversity_index  \
0                   14                  2                    0.499   
1                   13                  5                    0.298   
2                    3                  2                    0.174   
3                    8                  1                    0.263   
4                    7                  1                    0.298   

   total_delinquencies  credit_utilization_ratio  recent_inquiry_ratio     id  \
0                  1.0              12078.571429              0.142857  10000

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data[all_features], data[target], test_size=0.2, random_state=42
)


In [35]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

cat_model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0)
cat_model.fit(X_train, y_train)
y_pred = cat_model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_pred)
print(f"CatBoost AUC: {auc:.4f}")


CatBoost AUC: 0.7918


In [36]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

models = {
    'RandomForest': RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss'),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LightGBM': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.1, max_depth=6)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_test)[:,1]
    results[name] = roc_auc_score(y_test, y_pred)

print("Barcha model AUC natijalari:")
for name, auc in results.items():
    print(f"{name}: {auc:.4f}")


KeyboardInterrupt: 

In [38]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    data[all_features], data[target], test_size=0.2, random_state=42
)

# Modellar
models = {
    'RandomForest': RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss'),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LightGBM': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.1, max_depth=6)
}

results = {}

for name, model in models.items():
    # Pipeline: NaN to'ldirish + Standartizatsiya (LogisticRegression uchun)
    if name == 'LogisticRegression':
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('model', model)
        ])
    else:
        # Tree-based modellarda scaler shart emas, faqat imputer
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', model)
        ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict_proba(X_test)[:,1]
    results[name] = roc_auc_score(y_test, y_pred)

# Natijalarni chiqarish
print("Barcha model AUC natijalari:")
for name, auc in results.items():
    print(f"{name}: {auc:.4f}")


c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [02:37:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 3656, number of negative: 68343
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006962 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3602
[LightGBM] [Info] Number of data points in the train set: 71999, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.050778 -> initscore=-2.928169
[LightGBM] [Info] Start training from score -2.928169
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Barcha model AUC natijalari:
RandomForest: 0.7908
XGBoost: 0.7756
LogisticRegression: 0.7967
LightGBM: 0.7754


c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [39]:
import pandas as pd

# CSV faylni yuklash
data = pd.read_csv("merged.csv")

# Har bir guruh uchun korelatsiya matritsasi
groups = {
    "Group 1": group_1,
    "Group 2": group_2,
    "Group 3": group_3,
    "Group 4": group_4,
    "Group 5": group_5,
    "Group 6": group_6
}

for name, features in groups.items():
    corr_matrix = data[features].corr()
    print(f"Korelatsiya: {name} ({len(features)} columns)")
    print(corr_matrix, "\n")


Korelatsiya: Group 1 (7 columns)
                           credit_score  oldest_account_age_months  \
credit_score                   1.000000                   0.184977   
oldest_account_age_months      0.184977                   1.000000   
num_inquiries_6mo             -0.000733                   0.002845   
account_diversity_index        0.183210                   0.573298   
total_delinquencies           -0.040099                  -0.012151   
recent_inquiry_ratio          -0.097557                  -0.074790   
age                            0.443576                   0.335378   

                           num_inquiries_6mo  account_diversity_index  \
credit_score                       -0.000733                 0.183210   
oldest_account_age_months           0.002845                 0.573298   
num_inquiries_6mo                   1.000000                 0.004609   
account_diversity_index             0.004609                 1.000000   
total_delinquencies                -0.002

In [ ]:
import pandas as pd
import numpy as np

# Guruhlarni dictionary shaklida saqlaymiz
groups = {
    'Group 1': group_1,
    'Group 2': group_2,
    'Group 3': group_3,
    'Group 4': group_4,
    'Group 5': group_5,
    'Group 6': group_6
}

# Har bir guruh uchun eng past korelatsiyani topish
for name, features in groups.items():
    corr_matrix = data[features].corr()
    
    # Diagonalni tashlab eng past korelatsiya
    corr_matrix_values = corr_matrix.where(~np.eye(corr_matrix.shape[0], dtype=bool))
    
    # Eng past korelatsiya va ustunlar
    min_corr = corr_matrix_values.min().min()
    min_pair = corr_matrix_values.stack().idxmin()
    
    print(f"{name} - eng past korelatsiya: {min_corr:.4f} (ustunlar: {min_pair})")


Group 1 - eng past korelatsiya: -0.4904 (ustunlar: ('num_credit_accounts', 'credit_utilization_ratio'))
Group 2 - eng past korelatsiya: -0.1563 (ustunlar: ('annual_income', 'education'))
Group 3 - eng past korelatsiya: -0.4236 (ustunlar: ('credit_utilization', 'available_credit'))
Group 4 - eng past korelatsiya: -0.5759 (ustunlar: ('loan_to_annual_income', 'monthly_free_cash_flow'))
Group 5 - eng past korelatsiya: -0.9200 (ustunlar: ('interest_rate', 'loan_purpose_risk'))
Group 6 - eng past korelatsiya: -0.0068 (ustunlar: ('has_mobile_app', 'account_open_year'))


In [41]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import lightgbm as lgb

# Features va target ajratish
X = data[all_features]
y = data[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Categorical va numeric ustunlarni aniqlash
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# SMOTE bilan oversampling
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Base modellar
estimators = [
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('xgb', XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        use_label_encoder=False, eval_metric='logloss', random_state=42
    )),
    ('lgbm', lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42
    ))
]

# Stacking classifier
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', stacking_clf)
])

# Fit qilish
pipeline.fit(X_train_res, y_train_res)

# Test AUC
y_pred = pipeline.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred)
print(f"Test set AUC: {auc:.4f}")


ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [42]:
from sklearn.impute import SimpleImputer

# Numeric ustunlar uchun median bilan to'ldirish
num_imputer = SimpleImputer(strategy='median')

# Categorical ustunlar uchun mode bilan to'ldirish
cat_imputer = SimpleImputer(strategy='most_frequent')

# ColumnTransformer ichida imputer va scaler/encoder ishlatish
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', num_imputer), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', cat_imputer), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
    ]
)


In [46]:
from sklearn.impute import SimpleImputer

# Sonli ustunlar uchun o‘rtacha bilan to‘ldirish
num_imputer = SimpleImputer(strategy='mean')
X_train_num = num_imputer.fit_transform(X_train)
X_test_num = num_imputer.transform(X_test)


In [47]:
cat_cols = ['employment_type','loan_type_clean','secured_status','preferred_contact']
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])


In [49]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

estimators = [
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('rf', RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')),
    ('lgb', LGBMClassifier(n_estimators=500, learning_rate=0.1, max_depth=6))
]


In [50]:
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import roc_auc_score

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

# Modelni o'qitish
stack_model.fit(X_train_res, y_train_res)

# Test set bo'yicha AUC
y_pred_proba = stack_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Stacking model AUC: {auc:.4f}")


NameError: name 'X_train_res' is not defined